# Run PhosIrDesign In Colab

This notebook is a thin launcher for the PhosIrDesign repository workflow.

- The canonical entry point is `run.sh`; setup and workflow internals live under `environment/` and `scripts/workflows/`.
- The notebook does not reimplement training, prediction, figure generation, or screening logic.
- `data/ours.csv` is the post-screening synthesized experimental test set used for prediction exports.
- Run the cells from top to bottom in Google Colab or a standard Jupyter environment with internet access.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/zots0127/PhosIrDesign.git"
BRANCH = "main"
REPO_DIR = Path("PhosIrDesign")
RUN_COMMAND = "bash run.sh"

print({
    "REPO_URL": REPO_URL,
    "BRANCH": BRANCH,
    "REPO_DIR": str(REPO_DIR),
    "RUN_COMMAND": RUN_COMMAND,
})


In [ ]:
import subprocess

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "--depth", "1"], cwd=REPO_DIR, check=True)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"Repository ready at {REPO_DIR.resolve()}")
print(f"Active commit: {commit}")


In [ ]:
import os

print(f"Running: {RUN_COMMAND}")
subprocess.run(RUN_COMMAND, cwd=REPO_DIR, shell=True, check=True, env=os.environ.copy())


In [ ]:
output_dirs = sorted(REPO_DIR.glob("Project_Output*"), key=lambda path: path.stat().st_mtime, reverse=True)
artifact_names = [
    "workflow_summary.json",
    "model_comparison_detailed.csv",
    "virtual_predictions_all.csv",
    "test_predictions",
    "figures",
    "shap_analysis/shap_report.html",
]

selected_output_dir = None
if not output_dirs:
    print("No Project_Output* directories were found.")
else:
    scored_outputs = []
    for path in output_dirs:
        score = sum((path / artifact_name).exists() for artifact_name in artifact_names)
        scored_outputs.append((score, path.stat().st_mtime, path))
    scored_outputs.sort(reverse=True)
    selected_output_dir = scored_outputs[0][2]
    print(f"Selected output directory: {selected_output_dir}")
    print("\nRecent output directories:")
    for path in output_dirs[:5]:
        print(f"- {path}")

    print("\nRepresentative artifacts:")
    candidates = [selected_output_dir / artifact_name for artifact_name in artifact_names]
    for path in candidates:
        print(f"- {path}: {'FOUND' if path.exists() else 'missing'}")


In [ ]:
from IPython.display import Image, display

if selected_output_dir is not None:
    figure_candidates = [
        selected_output_dir / "figures" / "figure_c_wavelength_plqy.png",
        selected_output_dir / "figures" / "figure_h_virtual_screening.png",
        selected_output_dir / "figures" / "figure_i_ligand_analysis.png",
    ]
    for figure_path in figure_candidates:
        if figure_path.exists():
            print(f"Displaying {figure_path}")
            display(Image(filename=str(figure_path)))
else:
    print("No outputs available for figure preview.")
